# D1.3 · Agent-assisted detection engineering

**Function D — AI for SecOps → The SOC Analyst & Detection Engineer**  ·  *AI for Security*

Builds on **[D1.2 · Context that makes triage work](https://spbreed.github.io/cyber-commons/lessons/D1.2.html)**.

| | |
|---|---|
| Open-source tooling | Sigma, Wazuh |
| Open-weight models | Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

An agent can write and tune a detection far faster than you can, which means it can also ship a confident, wrong rule into production far faster than you can. The validation discipline is the whole of the value.

## 2 · The framework

```
   agent writes rule --> test corpus --> tuned rule --> production
                              ^
                       +------+-------+
                       | true positives from history |
                       | benign traffic that must    |
                       |   NOT fire                  |
                       +-----------------------------+

   the speed is real. so is the speed of shipping a wrong rule.
```

Using an agent to write detections is genuinely effective: it produces candidate
rules quickly, across more log sources than a human would attempt.

What it cannot supply is the judgement that decides whether a rule ships, because
that judgement depends on a cost the telemetry does not contain: **analyst
trust**. A rule with 5% precision is not 5% useful — it is negatively useful,
because it spends attention that the good rules need.

So the workflow is: the agent generates candidates, and a scoring step against
real historical telemetry decides which survive. The scoring step is the job, and
it is the part teams skip.

> **About the model in this notebook.** It runs offline against a deterministic
> replay so the lesson executes on a Kaggle kernel with no network. To run the
> same triage against a real open-weight model:
>
> ```bash
> ollama pull glm-4.6            # or kimi-k2, llama3.3
> export OPENAI_BASE_URL=http://localhost:11434/v1 OPENAI_API_KEY=ollama MODEL=glm-4.6
> ```

## 3 · Demo — five candidate rules for one concern

In [ ]:
import time
from dataclasses import dataclass

@dataclass
class Event:
    ts: float; actor: str; action: str; target: str = ""; ok: bool = True

now = time.time()
HISTORY  = [Event(now+i, "patch-agent", "http_get", "https://api.github.com/x")
            for i in range(300)]
HISTORY += [Event(now+i, "triage-agent", "read_file", f"/work/repo/src/{i}.py")
            for i in range(200)]
HISTORY += [Event(now+400, "patch-agent", "http_get",
                  "http://169.254.169.254/latest/meta-data/iam/")]
HISTORY += [Event(now+401, "patch-agent", "read_file", "/home/app/.aws/credentials")]
HISTORY += [Event(now+i, "svc-etl", "read_file", "/data/export.csv", ok=False)
            for i in range(20)]

TRUE_POSITIVES = {(now+400, "patch-agent"), (now+401, "patch-agent")}

CANDIDATES = {
 "R1 any http_get by an agent":
    lambda e: e.action == "http_get",
 "R2 http_get to a non-github host":
    lambda e: e.action == "http_get" and "api.github.com" not in e.target,
 "R3 link-local address":
    lambda e: "169.254." in e.target,
 "R4 any failed action":
    lambda e: not e.ok,
 "R5 credential path OR link-local":
    lambda e: "169.254." in e.target or "/.aws/" in e.target,
}
print(f"history: {len(HISTORY)} events, {len(TRUE_POSITIVES)} true positives")

In [ ]:
def score(rule, history, truth):
    fired = [e for e in history if rule(e)]
    tp = sum(1 for e in fired if (e.ts, e.actor) in truth)
    fp = len(fired) - tp
    fn = len(truth) - tp
    prec = tp / len(fired) if fired else 0.0
    rec  = tp / len(truth) if truth else 0.0
    return {"alerts": len(fired), "tp": tp, "fp": fp, "fn": fn,
            "precision": round(prec, 3), "recall": round(rec, 3),
            "alerts_per_tp": round(len(fired)/tp, 1) if tp else float("inf")}

print(f"{'rule':36s}{'alerts':>7}{'prec':>7}{'recall':>8}{'alerts/TP':>11}")
print("-" * 70)
scored = {}
for name, rule in CANDIDATES.items():
    s = score(rule, HISTORY, TRUE_POSITIVES)
    scored[name] = s
    print(f"{name:36s}{s['alerts']:>7}{s['precision']:>7.3f}{s['recall']:>8.3f}"
          f"{str(s['alerts_per_tp']):>11}")

## 4 · Where it breaks — every rule 'works'

All five detect something. R1 has perfect recall on http traffic and would put 301 alerts a day in the queue. R4 has 100% precision on nothing useful. The deployable set is decided by a threshold nobody writes down.

In [ ]:
MAX_ALERTS_PER_TP = 5          # the analyst-trust budget, made explicit
MIN_RECALL = 0.5

def deployable(s):
    reasons = []
    if s["tp"] == 0:                       reasons.append("no true positives")
    if s["alerts_per_tp"] > MAX_ALERTS_PER_TP:
        reasons.append(f"{s['alerts_per_tp']} alerts per true positive "
                       f"(budget {MAX_ALERTS_PER_TP})")
    if s["recall"] < MIN_RECALL:           reasons.append(f"recall {s['recall']} below {MIN_RECALL}")
    return (not reasons), reasons

for name, s in scored.items():
    ok, reasons = deployable(s)
    print(f"{'DEPLOY' if ok else 'REJECT':7s} {name}")
    for r in reasons: print(f"          · {r}")

## 5 · The control — generate many, score against history, ship few

In [ ]:
def workflow(candidates, history, truth):
    scored = {n: score(r, history, truth) for n, r in candidates.items()}
    shipped = {n: s for n, s in scored.items() if deployable(s)[0]}
    return {
      "generated": len(candidates),
      "shipped": len(shipped),
      "shipped_rules": sorted(shipped),
      "queue_impact_per_day": sum(s["alerts"] for s in shipped.values()),
      "coverage": round(max((s["recall"] for s in shipped.values()), default=0), 3),
    }
w = workflow(CANDIDATES, HISTORY, TRUE_POSITIVES)
for k, v in w.items(): print(f"{k:24s}{v}")

print("\nThe agent generated 5 rules in seconds. Scoring them against 521 real")
print("events took milliseconds and rejected 3. That scoring step is the job —")
print("without it, R1 ships and the SOC stops reading agent alerts within a week.")
assert w["shipped"] < w["generated"]
assert "R5 credential path OR link-local" in w["shipped_rules"]

## What you just proved

All five rules detect something. R1 fires 301 times for 1 true positive; R5 fires twice for 2 true positives with perfect precision and recall. The deployability check rejects the broad rules and the failed-action rule, shipping only the precise ones with a small daily queue impact.

## Your turn

Set your own alerts-per-true-positive budget and apply it to the rules already in production. Most SOCs discover that several long-standing rules would not pass the bar they would set today.

---

**Next → [D1.4 · Detection engineering *for* agents](https://spbreed.github.io/cyber-commons/lessons/D1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*